# Embedding Similarity Search with Rasteret

Find grain silos across Franklin County, Kansas using
[AlphaEarth Foundations Satellite Embedding dataset (AEF)](https://source.coop/tge-labs/aef)
produced by Google and Google DeepMind — 64-band int8 COGs at 10 m resolution.

This workflow is inspired by:
- Ujaval Gandhi’s GeoPython Tutorials post:
  [Large-Scale Embedding Similarity Search with xarray and Dask](https://www.geopythontutorials.com/notebooks/xarray_embeddings_similarity_search.html)
- Google Earth Engine community tutorial:
  [Satellite Embedding Similarity Search](https://developers.google.com/earth-engine/tutorials/community/satellite-embedding-05-similarity-search)

Data credits:
- County boundary data: **US Census Bureau, 2021 Cartographic Boundary Files**.
- Embeddings: **AlphaEarth Foundations Satellite Embedding dataset (produced by Google and Google DeepMind)**.

| Step | Rasteret API | What it does |
|---|---|---|
| Reference vector | `sample_points` | Extract embeddings at known grain-silo locations |
| Approach A | `get_xarray` | Dense mosaic -> similarity raster -> match polygons |
| Approach B | `get_gdf` | Per-record arrays -> similarity arrays -> match polygons |
| Approach C | `to_torchgeo_dataset` | Streaming 1024 px chips -> stitched similarity raster -> match polygons |


Attribution:

> "The AlphaEarth Foundations Satellite Embedding dataset is produced by Google and Google DeepMind."


---
## Setup

AEF stores embeddings as int8 with nodata = -128. The dequantization
formula recovers float32 values: `(v / 127.5)^2 * sign(v)`.

All three approaches share the same similarity math. The difference is how much
data they materialize and what shape they return:

- `get_xarray`: easiest dense raster for small-to-medium AOIs, highest memory.
- `get_gdf`: record-wise arrays; good when you want map-ready match polygons
  without holding one full mosaic.
- `to_torchgeo_dataset`: chip streaming; best mental model for training or
  inference loops where memory should stay bounded.

To keep the comparison fair, each approach finishes by producing high-similarity
match polygons using the same threshold.


In [ ]:
import os
import time

import geopandas as gpd
import numpy as np
import pandas as pd
from affine import Affine
from rasterio.features import geometry_mask, shapes
from shapely.geometry import Point, shape
from sklearn.metrics.pairwise import cosine_similarity

import rasteret

os.environ.setdefault("AWS_NO_SIGN_REQUEST", "YES")
COLLECTION_URI = "aef/v1-annual"

rasteret.set_options(progress=False)

ALL_BANDS = [f"A{i:02d}" for i in range(64)]
NODATA = -128
THRESHOLD = 0.95

REFERENCE_POINTS = [
    (-95.18616479385629, 38.54715519758577),
    (-95.34468619878159, 38.59339901996762),
    (-95.34280239688128, 38.56233059960432),
]


def dequantize(v: np.ndarray) -> np.ndarray:
    """AEF int8 -> float32. NODATA (-128) becomes NaN."""
    f = v.astype(np.float32)
    nd = (f == NODATA) | np.isnan(f)
    out = (f / 127.5) ** 2 * np.sign(f)
    out[nd] = np.nan
    return out


def cosine_similarity_map(cube: np.ndarray, ref: np.ndarray) -> np.ndarray:
    """Cosine similarity between each pixel and reference embedding via sklearn.

    cube : (C, H, W) int8 array — raw AEF embeddings
    ref  : (C,) float32 reference embedding (already dequantized)

    Returns (H, W) float32 array, NaN where any band is nodata or all-zero.
    """
    C, H, W = cube.shape
    flat = dequantize(cube.reshape(C, -1).T)  # (N, C)

    sim_flat = np.full(flat.shape[0], np.nan, dtype=np.float32)
    valid = np.isfinite(flat).all(axis=1)
    if np.any(valid):
        rows = flat[valid]
        nonzero = np.linalg.norm(rows, axis=1) > 0
        if np.any(nonzero):
            sim = cosine_similarity(rows[nonzero], ref.reshape(1, -1)).ravel()
            valid_idx = np.flatnonzero(valid)
            sim_flat[valid_idx[nonzero]] = sim.astype(np.float32)

    return sim_flat.reshape(H, W)


def affine_from_center_coords(x_coords: np.ndarray, y_coords: np.ndarray) -> Affine:
    """Build a rasterio Affine for a derived array using xarray center coords."""
    x_res = float(np.nanmedian(np.diff(x_coords)))
    y_res = float(np.nanmedian(np.diff(y_coords)))
    return Affine(
        x_res,
        0,
        float(x_coords[0] - x_res / 2),
        0,
        y_res,
        float(y_coords[0] - y_res / 2),
    )


def vectorize_matches(
    sim: np.ndarray,
    transform: Affine,
    crs: int | str,
    *,
    source: str,
    threshold: float = THRESHOLD,
) -> gpd.GeoDataFrame:
    """Turn a similarity raster into polygons for pixels above threshold."""
    hot = np.isfinite(sim) & (sim >= threshold)
    polygons = [
        shape(geom)
        for geom, val in shapes(hot.astype(np.uint8), mask=hot, transform=transform)
        if val == 1
    ]
    return gpd.GeoDataFrame(
        {
            "source": [source] * len(polygons),
            "match_id": np.arange(len(polygons), dtype=np.int32),
        },
        geometry=polygons,
        crs=crs,
    )


def print_similarity_stats(label: str, sim: np.ndarray) -> int:
    finite = sim[np.isfinite(sim)]
    print(
        f"{label}  min={finite.min():.4f}  mean={finite.mean():.4f}  "
        f"max={finite.max():.4f}  pixels={finite.size:,}"
    )
    return finite.size


timings: dict[str, float] = {}

## Load the AEF collection

Good news: the AEF Rasteret collection is already published for **2018–2024**
on **Source Cooperative** and **Hugging Face**.

For this notebook, Rasteret runs `rasteret.load("aef/v1-annual")`
for you so it can reopen the published Collection directly. The narrow
`index.parquet` stays in front for pushdown, so you do not need to prepare
any source data by hand.


In [ ]:
COUNTY_URL = "https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_500k.zip"
counties = gpd.read_file(COUNTY_URL)
franklin = counties[counties["GEOID"] == "20059"].to_crs(4326)
franklin_geom = franklin.geometry.iloc[0]
franklin_utm = franklin.to_crs(32615)
county_geom_utm = franklin_utm.geometry.iloc[0]

collection = rasteret.load(COLLECTION_URI, name="aef-franklin-2024")
collection.describe()

aef_franklin_2024 = collection.subset(
    bbox=tuple(franklin.total_bounds.tolist()),
    date_range=("2024-01-01", "2024-12-31"),
)
print(f"{len(aef_franklin_2024)} AEF tiles cover Franklin County in 2024")

---
## Reference embedding with `sample_points`

`sample_points` reads only the tiles containing the query points — no
full-AOI load. It returns a PyArrow table with one row per point and band.

For readability, we convert that small result to pandas, pivot it to a
`(3, 64)` matrix, dequantize, and average into one 64-dimensional reference
vector.


In [ ]:
t0 = time.perf_counter()

reference_points = [Point(lon, lat) for lon, lat in REFERENCE_POINTS]
reference_samples = aef_franklin_2024.sample_points(
    reference_points,
    bands=ALL_BANDS,
    geometry_crs=4326,
)

timings["ref_sample"] = time.perf_counter() - t0
print(
    f"sample_points: {reference_samples.num_rows} rows in {timings['ref_sample']:.1f}s"
)

# Pivot to one row per reference point and one column per AEF band.
reference_samples_df = reference_samples.to_pandas()
reference_wide = (
    reference_samples_df.pivot(index="point_index", columns="band", values="value")
    .reindex(columns=ALL_BANDS)
    .sort_index()
)

raw_reference_values = reference_wide.to_numpy()
reference_embeddings = [dequantize(row) for row in raw_reference_values]
reference_vector = np.nanmean(reference_embeddings, axis=0)
reference_norm = float(np.linalg.norm(reference_vector))

for point_index, embedding in enumerate(reference_embeddings):
    cosine_to_mean = float(
        np.dot(embedding, reference_vector)
        / (np.linalg.norm(embedding) * reference_norm)
    )
    print(f"  ref {point_index}: similarity to mean = {cosine_to_mean:.3f}")

---
## Approach A — `get_xarray` + dense map

`get_xarray` mosaics all overlapping tiles into one `xarray.Dataset` with
spatial coordinates and CRS metadata. This is the most convenient path when you
want a full raster-like surface and you have enough RAM.

> **Memory note:** For AEF (64 bands), `get_xarray` materializes the full AOI mosaic. For Franklin County this can peak around **20-24 GB RAM** even with `int8` source bands.

We stack bands into a `(64, H, W)` cube, compute one county-wide similarity map,
and then vectorize pixels above `THRESHOLD` into polygons.


In [ ]:
t0 = time.perf_counter()

xarray_dataset = aef_franklin_2024.get_xarray(
    geometries=franklin_geom,
    bands=ALL_BANDS,
    geometry_crs=4326,
)
timings["A_load"] = time.perf_counter() - t0

H, W = len(xarray_dataset.y), len(xarray_dataset.x)
print(
    f"mosaic shape: ({H}, {W})  bands: {len(ALL_BANDS)}  load: {timings['A_load']:.1f}s"
)

In [ ]:
t0 = time.perf_counter()

# to_dataarray() stacks all bands into a single (band, y, x) DataArray — no manual loop
xarray_band_cube = (
    xarray_dataset[ALL_BANDS].to_dataarray(dim="band").isel(time=0, geometry=0).values
)
similarity_from_xarray = cosine_similarity_map(xarray_band_cube, reference_vector)

timings["A_cosine"] = time.perf_counter() - t0
timings["A_total"] = timings["A_load"] + timings["A_cosine"]

# The similarity array is a derived NumPy array, so pass xarray's coordinate
# georeferencing to rasterio.shapes without changing Rasteret's row order.
xarray_y_coords = xarray_dataset.y.values
xarray_plot_origin = "lower" if xarray_y_coords[-1] > xarray_y_coords[0] else "upper"
xarray_transform = affine_from_center_coords(
    xarray_dataset.x.values,
    xarray_y_coords,
)
del xarray_dataset, xarray_band_cube
matches_from_xarray = vectorize_matches(
    similarity_from_xarray,
    xarray_transform,
    32615,
    source="A_get_xarray",
).to_crs(4326)

print_similarity_stats("similarity", similarity_from_xarray)
print(f"matches >= {THRESHOLD}: {len(matches_from_xarray)} polygons")
print(
    f"timing  load={timings['A_load']:.1f}s  "
    f"cosine+vector={timings['A_cosine']:.1f}s  "
    f"total={timings['A_total']:.1f}s"
)

---
## Approach B — `get_gdf` + record-wise matches

`get_gdf` returns a GeoDataFrame with one row per `(record, band)`. Each row
carries a 2-D NumPy array in the `data` column and the read window `transform`.
Records keep their native shapes — no full-county mosaic and no global resampling.

> **Memory note:** For this Franklin County setup with AEF 64 bands, `get_gdf` is typically around **~3 GB peak RAM** (lower than full `get_xarray`, usually higher than chip-wise TorchGeo streaming).

For a similarity-search workflow, this is a natural path: compute similarity per
record, threshold it, and vectorize only the matching pixels. You get map-ready
polygons without keeping a dense county-wide similarity raster.


In [ ]:
t0 = time.perf_counter()

record_arrays_gdf = aef_franklin_2024.get_gdf(
    geometries=franklin_geom,
    bands=ALL_BANDS,
    geometry_crs=4326,
    target_crs=32615,
)
timings["B_load"] = time.perf_counter() - t0

n_records = record_arrays_gdf["record_id"].nunique()
print(
    f"get_gdf: {len(record_arrays_gdf)} rows  "
    f"({n_records} records x {len(ALL_BANDS)} bands)  "
    f"load: {timings['B_load']:.1f}s"
)

In [ ]:
t0 = time.perf_counter()

total_gdf_pixels = 0
gdf_match_parts = []
for rid in record_arrays_gdf["record_id"].unique():
    rec = record_arrays_gdf[record_arrays_gdf["record_id"] == rid].sort_values("band")
    cube = np.stack([row["data"] for _, row in rec.iterrows()], axis=0)
    similarity_for_record = cosine_similarity_map(cube, reference_vector)

    total_gdf_pixels += print_similarity_stats(
        f"  record {rid}:", similarity_for_record
    )
    gdf_match_parts.append(
        vectorize_matches(
            similarity_for_record,
            rec["transform"].iloc[0],
            record_arrays_gdf.crs,
            source=f"B_get_gdf:{rid}",
        )
    )

matches_from_gdf = (
    gpd.GeoDataFrame(
        pd.concat(gdf_match_parts, ignore_index=True),
        geometry="geometry",
        crs=record_arrays_gdf.crs,
    ).to_crs(4326)
    if gdf_match_parts
    else gpd.GeoDataFrame(
        {"source": [], "match_id": []}, geometry=[], crs=record_arrays_gdf.crs
    ).to_crs(4326)
)

timings["B_cosine"] = time.perf_counter() - t0
timings["B_total"] = timings["B_load"] + timings["B_cosine"]
del record_arrays_gdf

print(f"\ntotal pixels: {total_gdf_pixels:,}")
print(f"matches >= {THRESHOLD}: {len(matches_from_gdf)} polygons")
print(
    f"timing  load={timings['B_load']:.1f}s  "
    f"cosine+vector={timings['B_cosine']:.1f}s  "
    f"total={timings['B_total']:.1f}s"
)

---
## Approach C — TorchGeo streaming

`to_torchgeo_dataset` wraps the collection as a TorchGeo `GeoDataset`. A
`GridGeoSampler` then reads fixed-size chips, so memory stays bounded and the
code looks like an ML inference loop.

Here we stitch chip predictions with `notebooks/utils_stitching.py`, mask the
result to Franklin County, and vectorize the same `THRESHOLD` matches.

> **Memory note:** In this Franklin County setup, TorchGeo chip-wise loading is typically much lower peak RAM (about **3-5 GB**) than full-AOI `get_xarray` materialization.


In [ ]:
from torchgeo.samplers import GridGeoSampler

try:
    from notebooks.utils_stitching import stitch_prediction_tiles
except ImportError:
    from utils_stitching import stitch_prediction_tiles

CHIP_PX = 1024

t0 = time.perf_counter()

torchgeo_dataset = aef_franklin_2024.to_torchgeo_dataset(bands=ALL_BANDS)
grid_sampler = GridGeoSampler(
    torchgeo_dataset, size=CHIP_PX, stride=CHIP_PX, roi=county_geom_utm
)

res_x, res_y = torchgeo_dataset.res
roi_xmin, roi_ymin, roi_xmax, roi_ymax = county_geom_utm.bounds
out_w = round((roi_xmax - roi_xmin) / res_x)
out_h = round((roi_ymax - roi_ymin) / res_y)
n_chips = len(grid_sampler)

print(f"{n_chips} chips ({CHIP_PX}x{CHIP_PX} px)  output grid: ({out_h}, {out_w})")

chip_similarity_tiles = []
skipped = 0
for i, query in enumerate(grid_sampler):
    try:
        sample = torchgeo_dataset[query]
    except Exception:
        skipped += 1
        continue

    chip = sample["image"].numpy().astype(np.float32)
    chip_sim = cosine_similarity_map(chip, reference_vector)
    chip_similarity_tiles.append(
        {"prediction": chip_sim, "transform": sample["transform"].numpy()}
    )

    elapsed = time.perf_counter() - t0
    print(f"  chip {i + 1}/{n_chips}  ({elapsed:.0f}s)", end="\r")

similarity_from_torchgeo = stitch_prediction_tiles(
    chip_similarity_tiles,
    roi_bounds=(roi_xmin, roi_ymin, roi_xmax, roi_ymax),
    res=(res_x, res_y),
    reducer="overwrite",
    out_shape=(out_h, out_w),
)

# Mask to county polygon for apples-to-apples comparison
out_transform = Affine(res_x, 0, roi_xmin, 0, -res_y, roi_ymax)
county_mask = geometry_mask(
    [county_geom_utm],
    out_shape=(out_h, out_w),
    transform=out_transform,
    invert=True,
)
similarity_from_torchgeo[~county_mask] = np.nan

timings["C_total"] = time.perf_counter() - t0
torchgeo_dataset.close()

matches_from_torchgeo = vectorize_matches(
    similarity_from_torchgeo, out_transform, 32615, source="C_torchgeo"
).to_crs(4326)

print_similarity_stats("\nsimilarity", similarity_from_torchgeo)
print(f"matches >= {THRESHOLD}: {len(matches_from_torchgeo)} polygons")
print(
    f"timing  total={timings['C_total']:.1f}s  ({n_chips} chips, "
    f"used={len(chip_similarity_tiles)}, skipped={skipped})"
)

---
## Results

All three approaches now finish with comparable match polygons, but they are not
expected to be pixel-identical:

- **A / `get_xarray`**: full similarity raster first, then polygons. This is the
  easiest path to reason about but uses the most memory.
- **B / `get_gdf`**: record-wise similarity arrays, then polygons. This is a good
  fit for “find similar pixels/regions” because the final product is vector
  regions and we do not need to keep a full mosaic.
- **C / `to_torchgeo_dataset`**: streaming chips, stitched similarity raster,
  then polygons. This is closest to a production inference loop.

Small differences are normal. `get_xarray` uses a dense mosaic grid, `get_gdf`
uses each record/read window grid, and TorchGeo uses the sampler chip grid. At a
fixed threshold, pixels near the cutoff can flip and polygon boundaries/counts
can differ slightly.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)

for ax, arr, label, origin in [
    (
        axes[0],
        similarity_from_xarray,
        f"A  get_xarray ({timings['A_total']:.0f}s)",
        xarray_plot_origin,
    ),
    (
        axes[1],
        similarity_from_torchgeo,
        f"C  TorchGeo streaming ({timings['C_total']:.0f}s)",
        "upper",
    ),
]:
    im = ax.imshow(arr, cmap="RdYlGn", vmin=0.4, vmax=1.0, origin=origin)
    ax.set_title(label)
    ax.set_axis_off()

plt.colorbar(im, ax=axes, shrink=0.7, label="cosine similarity")
plt.suptitle("Dense similarity surfaces — Franklin County, Kansas", fontsize=14)
plt.show()

print(f"A polygons: {len(matches_from_xarray):,}")
print(f"B polygons: {len(matches_from_gdf):,}")
print(f"C polygons: {len(matches_from_torchgeo):,}")

### Map the high-similarity regions

For the final map we use **Approach B** (`get_gdf`) by default. It is the most
natural output for this task: we are looking for similar pixels/regions, not a
full raster product.

`matches_from_xarray` and `matches_from_torchgeo` are also available if you want
to swap the layer and compare the outputs. Treat them as comparable access-path
outputs, not as a pixel-perfect equivalence test.


In [ ]:
# Use the record-wise get_gdf result for the final map layer.
# Swap to matches_from_xarray or matches_from_torchgeo here to compare the other approaches.
matches = matches_from_gdf.copy()
matches["area_m2"] = matches.to_crs(32615).area

for label, layer in [
    ("A get_xarray", matches_from_xarray),
    ("B get_gdf", matches_from_gdf),
    ("C TorchGeo", matches_from_torchgeo),
]:
    print(f"{label:14s}: {len(layer):,} regions with similarity >= {THRESHOLD}")

In [ ]:
import branca.colormap as cm
import folium

matches["area_m2"] = matches.to_crs(32615).area
vmin = float(matches["area_m2"].min()) if len(matches) else 0.0
vmax = float(matches["area_m2"].max()) if len(matches) else 1.0
if vmax <= vmin:
    vmax = vmin + 1.0
cmap = cm.linear.YlOrRd_09.scale(vmin, vmax)

center = franklin.geometry.iloc[0].centroid

m = folium.Map(
    location=[float(center.y), float(center.x)],
    zoom_start=9,
    tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google",
)

folium.GeoJson(
    franklin.__geo_interface__,
    name="Franklin County",
    style_function=lambda _: {
        "color": "#9aa0a6",
        "weight": 2,
        "fillColor": "#9aa0a6",
        "fillOpacity": 0.05,
    },
).add_to(m)

for _, row in matches.iterrows():
    color = cmap(float(row["area_m2"]))
    folium.GeoJson(
        row.geometry.__geo_interface__,
        style_function=lambda _, color=color: {
            "color": color,
            "weight": 1.5,
            "fillColor": color,
            "fillOpacity": 0.75,
        },
    ).add_to(m)

cmap.caption = "Match region area (m²)"
cmap.add_to(m)
m

---
## Timing comparison

In [ ]:
print(f"{'':32s} {'Load':>8s} {'Cosine':>8s} {'Total':>8s}")
print(f"{'---':32s} {'---':>8s} {'---':>8s} {'---':>8s}")
print(
    f"{'A  get_xarray + sklearn':32s} "
    f"{timings['A_load']:7.0f}s {timings['A_cosine']:7.0f}s "
    f"{timings['A_total']:7.0f}s"
)
print(
    f"{'B  get_gdf + sklearn':32s} "
    f"{timings['B_load']:7.0f}s {timings['B_cosine']:7.0f}s "
    f"{timings['B_total']:7.0f}s"
)
print(
    f"{'C  TorchGeo streaming':32s} "
    f"{'--':>8s} {'--':>8s} "
    f"{timings['C_total']:7.0f}s"
)

---
## Summary

| API | Best use | Map output in this notebook | Memory (Franklin, AEF 64 bands) |
|---|---|---|---|
| `sample_points` | Build reference vectors from known coordinates | Reference vector only | Tiny (MB-scale) |
| `get_xarray` | Dense AOI-wide raster analysis | `matches_from_xarray` from full similarity raster | Highest (~20-24 GB peak) |
| `get_gdf` | Similar-pixel / similar-region search | `matches_from_gdf`, used by final Folium map | Medium (~3 GB peak) |
| `to_torchgeo_dataset` | Streaming chips for training/inference loops | `matches_from_torchgeo` from stitched chip predictions | Lowest (~3-5 GB peak) |

The three approaches answer the same question with the same reference vector and
similarity threshold. They differ in data shape, memory behavior, and output
grid, so their polygons should be similar but not necessarily identical:

- Use `get_xarray` when a dense raster is the product you want.
- Use `get_gdf` when you want similar regions as vectors and do not need to keep
  the full mosaic.
- Use `to_torchgeo_dataset` when your next step is a chip-wise model or bounded
  memory inference loop.
